
# Procesamiento de la Carta Marina de Córdoba 2023

## Descripción

A diferencia de 2013, 2015, 2017 y 2019, para 2023 se desarrolla un parser específico desde cero, debido al cambio sustancial en la estructura de la Carta Marina, especialmente por los registros de establecimientos distribuidos en múltiples líneas.

En esta etapa el trabajo se concentra exclusivamente en la Carta Marina 2023.

## Flujo de trabajo


1. PDF
2. TXT (pdftotext -layout)
3. CSV de mesas/escuelas/electores


**Productos esperados**

- `LugaresDeVotacion-elecciones-2023.pdf`
- `carta-marina-cordoba-2023.txt`
- `escuelas-elecciones-2023-cordoba.csv`



0. Dependencias

In [1]:
from google.colab import files
import pandas as pd
from pathlib import Path
import re

In [2]:
!apt-get update -qq
!apt-get install -y -qq poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


#1. Carga del PDF

In [3]:
archivo = files.upload()

nombre_archivo = next(iter(archivo))

print(f"Archivo cargado: {nombre_archivo}")


Saving 2023.pdf to 2023.pdf
Archivo cargado: 2023.pdf


#2. PDF a TXT

In [4]:
nombre_txt = Path(nombre_archivo).with_suffix(".txt")

!pdftotext -layout "{nombre_archivo}" "{nombre_txt}"

print(f"Archivo generado: {nombre_txt}")

Archivo generado: 2023.txt


#3. TXT a CSV

##Inspeccion del TXT

In [5]:
path = f"/content/{nombre_txt}"

with open(path, "r", encoding="utf-8") as f:
    raw = f.read()

lines = raw.splitlines()

print(f"Cantidad total de líneas: {len(lines)}")
print("\nPrimeras 10 líneas:\n")

for i, linea in enumerate(lines[:10], start=1):
    print(i, repr(linea))
print ("...")
print ("...")
print ("...")
print ("...")
for i, linea in enumerate(raw.splitlines()[-10:], start=len(raw.splitlines())-9):
    print(i, repr(linea))

Cantidad total de líneas: 7322

Primeras 10 líneas:

1 '                       TRIBUNAL ELECTORAL PROVINCIAL - PROVINCIA DE CÓRDOBA'
2 '                            Elecciones Provinciales del 25 de Junio de 2023'
3 '                                Establecimientos, Mesas y Electores'
4 ''
5 ''
6 '                                     1 - Capital'
7 '                               1 - SECCIONAL PRIMERA'
8 'Establecimiento                                  Mesas       Tipo Mesas       Electores'
9 ''
10 'ESC NUESTRA SEÑORA DEL HUERTO - CALLE BELGRANO   1-8         Mixto            2748'
...
...
...
...
7313 'VIAMONTE'
7314 'Resumen Circuito   Establecimientos 1       Mesas: 5                Electores: 1686'
7315 ''
7316 'Resumen Sección    Establecimientos 55      Mesas: 291              Electores: 96227'
7317 ''
7318 ''
7319 ''
7320 ''
7321 '                                                                       Página 120 de 120'
7322 ''


In [6]:
print("Saltos de línea \\n:", raw.count("\n"))
print("Saltos de página \\f:", raw.count("\f"))

print("Con split('\\n'):", len(raw.split("\n")))
print("Con splitlines():", len(raw.splitlines()))

Saltos de línea \n: 7202
Saltos de página \f: 120
Con split('\n'): 7203
Con splitlines(): 7322


##Ciclo for
```
Leer línea
   │
   ├── ¿Es encabezado, página o línea vacía?
   │       └── Ignorar
   │
   ├── ¿Es una sección?
   │       └── Actualizar sección
   │
   ├── ¿Es un circuito?
   │       └── Actualizar circuito
   │
   ├── ¿Contiene un rango de mesas?  ← NUEVA ESCUELA
   │       │
   │       ├── Si había una escuela pendiente → guardarla
   │       │
   │       ├── Extraer rango de mesas
   │       ├── Extraer tipo de mesas
   │       ├── Extraer electores
   │       └── Guardar temporalmente nombre/domicilio
   │
   ├── ¿Hay una escuela pendiente?
   │       └── Agregar la línea al nombre/domicilio
   │
   └── ¿Es "Resumen Circuito"?
           ├── Guardar escuela pendiente
           └── Cerrar circuito

```



###Inicialización de variables

In [57]:
# Sección electoral actual
seccion_nro = 0
seccion_name = ""
esperando_seccion = True

# Circuito electoral actual
circuito_nro = ""
circuito_name = ""

# Lista donde se guardarán los establecimientos procesados
escuelas = []

# Registro temporal de la escuela que se está procesando.
# Permite acumular establecimientos que ocupan varias líneas.
escuela_actual = None

# Contador de líneas procesadas
cnt = 0

In [58]:
for linea in lines:
    cnt += 1

    linea_limpia = linea.strip()

    # Ignorar líneas vacías
    if linea_limpia == "":
        continue

    # Ignorar encabezados y elementos que no pertenecen a los establecimientos
    if linea_limpia.startswith("Resumen Circuito"):
        continue

    if linea_limpia.startswith("Establecimiento"):
        continue

    if "TRIBUNAL ELECTORAL PROVINCIAL" in linea_limpia:
        continue

    if "Elecciones Provinciales" in linea_limpia:
        continue

    if "Establecimientos, Mesas y Electores" in linea_limpia:
        continue

    if "Página" in linea_limpia:
        continue

    # "Mixtos" puede aparecer solo en la línea siguiente
    # de un registro "Extr y Nac."
    if linea_limpia == "Mixtos":
        continue

    # Detectar que terminó una sección
    if linea_limpia.startswith("Resumen Sección"):
        esperando_seccion = True
        continue

    # Detectar nueva sección
    if esperando_seccion:
        match_seccion = re.match(r"^(\d+)\s*-\s*(.+)$", linea_limpia)

        if match_seccion:
            seccion_nro = int(match_seccion.group(1))
            seccion_name = match_seccion.group(2).strip()
            esperando_seccion = False
            continue

    # Detectar circuito
    match_circuito = re.match(r"^(\d+[A-Z]?)\s*-\s*(.+)$", linea_limpia)

    if match_circuito:
        circuito_nro = match_circuito.group(1)
        circuito_name = match_circuito.group(2).strip()
        continue

    # Buscar rango de mesas seguido de alguno de los tipos conocidos
    match_rango = re.search(r"\b(\d+)\s*-\s*(\d+)\s+(Mixto|Ext Mixto|Extr y Nac\.)\s+(\d+)\b", linea)

    # Detectar el inicio de una nueva escuela
    if match_rango:

        # Guardar la escuela anterior antes de comenzar una nueva
        if escuela_actual is not None:
            escuelas.append(escuela_actual)

        mesa_desde = int(match_rango.group(1))
        mesa_hasta = int(match_rango.group(2))
        cant_mesas = mesa_hasta - mesa_desde + 1

        tipo_detectado = match_rango.group(3)

        if tipo_detectado == "Extr y Nac.":
            tipo_mesas = "Extr y Nac. Mixtos"
        else:
            tipo_mesas = tipo_detectado

        electores = int(match_rango.group(4))

        escuela_actual = {
            "seccion_nro": seccion_nro,
            "seccion_name": seccion_name,
            "circuito_nro": circuito_nro,
            "circuito_name": circuito_name,
            "escuela": linea[:match_rango.start()].strip(),
            "cant_mesas": cant_mesas,
            "desde": mesa_desde,
            "hasta": mesa_hasta,
            "tipo_mesas": tipo_mesas,
            "electores": electores
        }

        continue

    # Si no comienza una nueva escuela, agregar la línea a la escuela actual
    if escuela_actual is not None:
        escuela_actual["escuela"] += " " + linea_limpia

# Guardar la última escuela del archivo
if escuela_actual is not None:
    escuelas.append(escuela_actual)

## Correcciones de inconsistencias de la fuente

Durante la validación se detectaron algunos rangos de mesas inconsistentes en el documento original.  
Las correcciones se aplican después del parsing para mantener separada la extracción de datos de las modificaciones realizadas sobre la fuente.

Los casos se identificaron mediante la continuidad de la numeración, los rangos de establecimientos vecinos y los resúmenes de circuito.

In [43]:
correcciones = {
    ("183", 6152, 6227): (6220, 6227),
    ("194", 6256, 6600): (6599, 6600),
    ("245", 6967, 7073): (7066, 7073),
    ("361", 8722, 8746): (8722, 8729),
    ("364", 8729, 8744): (8738, 8744)
}

for escuela in escuelas:
    clave = (escuela["circuito_nro"], escuela["desde"], escuela["hasta"])

    if clave in correcciones:
        nuevo_desde, nuevo_hasta = correcciones[clave]

        escuela["desde"] = nuevo_desde
        escuela["hasta"] = nuevo_hasta
        escuela["cant_mesas"] = nuevo_hasta - nuevo_desde + 1

In [44]:
print(f"Escuelas detectadas: {len(escuelas)}")
print(f"Mesas totales: {sum(e['cant_mesas'] for e in escuelas)}")
print(f"Electores totales: {sum(e['electores'] for e in escuelas)}")

Escuelas detectadas: 1586
Mesas totales: 9057
Electores totales: 3051544


In [45]:
mesas = []

for escuela in escuelas:
    mesas.extend(range(escuela["desde"], escuela["hasta"] + 1))

from collections import Counter

conteo = Counter(mesas)
duplicadas = [mesa for mesa, cantidad in conteo.items() if cantidad > 1]

print(f"Mesas sumadas: {len(mesas)}")
print(f"Mesas únicas: {len(set(mesas))}")
print(f"Mesas duplicadas: {len(mesas) - len(set(mesas))}")
print(f"Números de mesa duplicados: {len(duplicadas)}")
print(f"Mesa máxima: {max(mesas)}")

Mesas sumadas: 9057
Mesas únicas: 9056
Mesas duplicadas: 1
Números de mesa duplicados: 1
Mesa máxima: 9060


In [46]:
# Mesa duplicada
duplicadas = [mesa for mesa, cantidad in conteo.items() if cantidad > 1]

print("Duplicadas:", duplicadas)

for mesa in duplicadas:
    print(f"\nMesa {mesa}")
    for escuela in escuelas:
        if escuela["desde"] <= mesa <= escuela["hasta"]:
            print(escuela)

# Mesas faltantes
faltantes = sorted(set(range(1, 9061)) - set(mesas))

print("\nMesas faltantes:", faltantes)

Duplicadas: [4547]

Mesa 4547
{'seccion_nro': 6, 'seccion_name': 'General San Martín', 'circuito_nro': '88', 'circuito_name': 'TIO PUJIO', 'escuela': 'IPEM N°172 JOSE HERNANDEZ - BV PERON (EX ENTRE RIOS) 66 - TIO PUJIO', 'cant_mesas': 6, 'desde': 4544, 'hasta': 4549, 'tipo_mesas': 'Mixto', 'electores': 1138}
{'seccion_nro': 6, 'seccion_name': 'General San Martín', 'circuito_nro': '88', 'circuito_name': 'TIO PUJIO', 'escuela': 'IPEM N°172 JOSE HERNANDEZ - BV PERON (EX ENTRE RIOS) 66 - TIO PUJIO                                               Mixtos', 'cant_mesas': 1, 'desde': 4547, 'hasta': 4547, 'tipo_mesas': 'Extr y Nac. Mixtos', 'electores': 348}

Mesas faltantes: [6152, 6256, 6967, 8746]


#Dataframe

In [47]:
df_escuelas = pd.DataFrame(escuelas)

display(df_escuelas.head())
display(df_escuelas.tail())

print(df_escuelas.shape)

,seccion_nro,seccion_name,circuito_nro,circuito_name,escuela,cant_mesas,desde,hasta,tipo_mesas,electores
0,1,Capital,1,SECCIONAL PRIMERA,ESC NUESTRA SEÑORA DEL HUERTO - CALLE BELGRANO...,8,1,8,Mixto,2748
1,1,Capital,1,SECCIONAL PRIMERA,"ESC NACIONAL DE MONSERRAT - OBISPO TREJO 294,C...",9,9,17,Mixto,3101
2,1,Capital,1,SECCIONAL PRIMERA,ESC SANTA TERESA DE JESUS - CALLE OBISPO TREJO...,8,18,25,Mixto,2750
3,1,Capital,1,SECCIONAL PRIMERA,INST INMACULADO CORAZON DE MARIA - CALLE ROSAR...,9,26,34,Mixto,3086
4,1,Capital,1,SECCIONAL PRIMERA,INST SEC MONSEÑOR DE ANDREA - VELEZ SARSFIELD ...,4,35,38,Mixto,1374


,seccion_nro,seccion_name,circuito_nro,circuito_name,escuela,cant_mesas,desde,hasta,tipo_mesas,electores
1581,26,Unión,397,SAN ANTONIO,ESC.PROV.P.PIZZURNO - MAESTRO FERNANDEZ 538 - ...,6,9037,9042,Mixto,1615
1582,26,Unión,400,SAN MARCOS SUD,"INST JOSE DE SAN MARTIN - ENTRE RIOS 977,SUD -...",5,9043,9047,Mixto,1644
1583,26,Unión,400,SAN MARCOS SUD,ESC PROV M BUCHARDO - PJE AURORA LLANOS DE BUS...,4,9048,9051,Mixto,1320
1584,26,Unión,401,SANTA MARIA,ESC PROV PAULA ALBARRACIN - LOS FRESNOS 290 - ...,1,9052,9052,Mixto,309
1585,26,Unión,402,VIAMONTE,INST JUAN B ALBERDI - AVELLANEDA 182 - VIAMONTE,5,9053,9057,Mixto,1686


(1586, 10)


In [48]:
df_escuelas.sample(20, random_state=42)

,seccion_nro,seccion_name,circuito_nro,circuito_name,escuela,cant_mesas,desde,hasta,tipo_mesas,electores
468,2,Calamuchita,20A,VILLA GENERAL BELGRANO,ESC.GRAL J. DE SAN MARTIN - AV NICARAGUA 566 -...,7,3341,3347,Mixto,2393
332,1,Capital,12H,LOS CERVECEROS,"ESC MADRE TERESA DE CALCUTA - PALAMARA 2900,B°...",8,2470,2477,Mixto,2771
946,12,Punilla,158,TANTI,"IPEM N°84 J VOCOS LESCANO - SAN FRANCISCO 33,B...",1,5904,5904,Extr y Nac. Mixtos,341
380,1,Capital,13J,VILLA AZALAIS,"ESC GDOR H DEL CASTILLO - GELLY Y OBES 3110,B°...",8,2805,2812,Mixto,2798
99,1,Capital,7B,ALTA CORDOBA,INST NUESTRA MADRE DE LA MERCE - M FRAGUEIRO 2...,8,729,736,Mixto,2800
1534,26,Unión,375,BELL VILLE,ESC.NORMAL F.ALCORTA - 25 DE MAYO 135 - BELL V...,9,8792,8800,Mixto,3149
1118,15,Río Seco,226A,PUESTO DE CASTRO,ESC.PROV.D.F.SARMIENTO - DOMINGO FAUSTINO SARM...,2,6754,6755,Mixto,429
1030,13,Río Cuarto,188,RIO CUARTO,IPEM N° 27 DR RENE FAVALORO - 11 DE NOVIEMBRE,7,6347,6353,Mixto,2440
939,12,Punilla,157,SANTA MARIA,"ESC JUAN BAUTISTA AZOPARDO - SARMIENTO 59,B° V...",7,5862,5868,Mixto,2425
303,1,Capital,12B,COLON,"INST MANUEL BELGRANO - LUIS BRAILE 2384,B°RIVA...",8,2254,2261,Mixto,2789


In [49]:
print(f"Registros: {len(df_escuelas)}")
print(f"Secciones: {df_escuelas['seccion_nro'].nunique()}")
print(f"Circuitos: {df_escuelas['circuito_nro'].nunique()}")
print(f"Electores: {df_escuelas['electores'].sum()}")

print("\nNulos por columna:")
print(df_escuelas.isnull().sum())

Registros: 1586
Secciones: 26
Circuitos: 643
Electores: 3051544

Nulos por columna:
seccion_nro      0
seccion_name     0
circuito_nro     0
circuito_name    0
escuela          0
cant_mesas       0
desde            0
hasta            0
tipo_mesas       0
electores        0
dtype: int64


In [51]:
duplicados = df_escuelas[df_escuelas.duplicated(keep=False)]

print(f"Registros duplicados: {len(duplicados)}")
duplicados

Registros duplicados: 0


,seccion_nro,seccion_name,circuito_nro,circuito_name,escuela,cant_mesas,desde,hasta,tipo_mesas,electores


In [59]:
df_escuelas = df_escuelas[
    [
        "seccion_nro",
        "seccion_name",
        "circuito_nro",
        "circuito_name",
        "escuela",
        "cant_mesas",
        "desde",
        "hasta",
        "tipo_mesas",
        "electores"
    ]
]

df_escuelas.head()

,seccion_nro,seccion_name,circuito_nro,circuito_name,escuela,cant_mesas,desde,hasta,tipo_mesas,electores
0,1,Capital,1,SECCIONAL PRIMERA,ESC NUESTRA SEÑORA DEL HUERTO - CALLE BELGRANO...,8,1,8,Mixto,2748
1,1,Capital,1,SECCIONAL PRIMERA,"ESC NACIONAL DE MONSERRAT - OBISPO TREJO 294,C...",9,9,17,Mixto,3101
2,1,Capital,1,SECCIONAL PRIMERA,ESC SANTA TERESA DE JESUS - CALLE OBISPO TREJO...,8,18,25,Mixto,2750
3,1,Capital,1,SECCIONAL PRIMERA,INST INMACULADO CORAZON DE MARIA - CALLE ROSAR...,9,26,34,Mixto,3086
4,1,Capital,1,SECCIONAL PRIMERA,INST SEC MONSEÑOR DE ANDREA - VELEZ SARSFIELD ...,4,35,38,Mixto,1374


In [56]:
output_path = "escuelas-elecciones-2023-cordoba.csv"

df_escuelas.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivo exportado: {output_path}")

Archivo exportado: escuelas-elecciones-2023-cordoba.csv
